# Kotoba Multilingual Speech-to-Text (STT) Streaming - Real-time Transcription in Eight Languages on SageMaker

This notebook deploys the **Kotoba Multilingual Speech-to-Text (STT) Streaming** model package from AWS Marketplace and transcribes audio over a SageMaker bidirectional stream, end to end.

**Product**: Kotoba Multilingual Speech-to-Text (STT) Streaming (version `v1.0`)
**Seller**: Kotoba AI

## Key Features

- Real-time transcription in eight languages (`ja`, `en`, `zh`, `zh-tw`, `ko`, `yue`, `es`, `vi`), about 0.8 s behind the audio
- For Japanese: katakana person names and clean transcripts without fillers (`style_preference`)
- Keyword biasing for up to 1,000 terms
- Turn detection on by default (sentence-aware `semantic_vad`), or `server_vad`, or off
- `pcm16`, `float32`, `mulaw` / `twilio` and Ogg/Opus input at any sample rate
- Billed per hour of input audio

Not available in this version: timestamps and speaker diarization.

The full protocol reference is in [README.md](./README.md).

---

## IMPORTANT: Bidirectional streaming only

The endpoint is **not** a standard REST endpoint:

- `POST /invocations` does **not** transcribe audio
- Batch transform jobs are **not** supported
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]==0.13.1` and `smithy-http[awscrt]` (this notebook uses the 0.13 client API)

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba Multilingual Speech-to-Text (STT) Streaming

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateModel",
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteModel",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:DescribeEndpointConfig",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Inference](#4.-Perform-Real-time-Inference)
5. [Troubleshooting](#5.-Troubleshooting)
6. [Clean-up](#6.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the model package:

1. Open the [listing of Kotoba Multilingual Speech-to-Text (STT) Streaming](https://aws.amazon.com/marketplace/pp/prodview-ven4djetklnes) on AWS Marketplace
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first — the Python version check in 2.2 doesn't require these packages, but every cell after that does.

In [ ]:
# The crt extra is needed for credentials from `aws login`.
!pip install -q --upgrade "boto3[crt]" numpy soundfile
!pip install -q --upgrade "sagemaker>=2.0,<3"
# The HTTP/2 SDK is not yet 1.0 and changes its API between minor versions: this notebook
# uses 0.13. smithy-http[awscrt] adds the HTTP/2 transport that bidirectional streaming needs.
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]==0.13.1" "smithy-http[awscrt]"


### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")


### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
from pathlib import Path

import boto3
import numpy as np
import sagemaker
import soundfile as sf

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.aio.client import AsyncSageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.aio.config import AsyncSageMakerRuntimeHTTP2Config
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_http.aio.crt import AWSCRTHTTPClient

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` will pick up the attached role automatically.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.


In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")


### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# Replace the placeholder with the Model Package ARN of your AWS Marketplace subscription (section 1, step 5).
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )


## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

The model runs on one GPU. Supported instance types:

| Instance types | GPU | Concurrent sessions per instance |
|----------------|-----|----------------------------------|
| `ml.g6e.4xlarge`, `ml.g6e.8xlarge`, `ml.g6e.16xlarge` | 1x NVIDIA L40S | up to 72 (measured on `ml.g6e.4xlarge`) |
| `ml.g7e.4xlarge`, `ml.g7e.8xlarge` | 1x NVIDIA RTX PRO 6000 Blackwell | up to 256 |
| `ml.p5.4xlarge` | 1x NVIDIA H100 | up to 320 |

Smaller `ml.g6e` sizes are not supported: their CPU cannot keep up at this load.

A connection over capacity is refused before the stream opens; raise the instance count for more sessions. The first start takes about 5 minutes.

In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-multilingual-stt-{int(time.time())}"
INSTANCE_TYPE = "ml.g6e.4xlarge"  # or ml.g6e.8xlarge / ml.g6e.16xlarge (L40S), ml.g7e.4xlarge / ml.g7e.8xlarge (RTX PRO 6000), ml.p5.4xlarge (H100)

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")


In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes 5-10 minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")


## 4. Perform Real-time Inference

### 4.1 Connection Protocol

The endpoint is reached through SageMaker's bidirectional-streaming transport:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Container
```

Standard WebSocket libraries **cannot** connect to SageMaker endpoints directly; use `aws-sdk-python[sagemaker-runtime-http2]` with model invocation path `invocations-bidirectional-stream`.

### 4.2 Protocol Flow

```
Client                                         Server
  |<----- transcription_session.created -------|
  |------ transcription_session.update ------->|   (once)
  |<----- transcription_session.updated -------|
  |------ input_audio_buffer.append ---------->|   (repeated, streaming)
  |<----- conversation.item.created -----------|   (once)
  |<----- conversation.item.input_audio_transcription.delta --------|   (repeated)
  |<----- conversation.item.input_audio_transcription.completed ----|   (per turn, with turn detection)
  |------ input_audio_buffer.commit ---------->|
  |<----- input_audio_buffer.committed --------|   (then the server closes the stream)
```

### 4.3 Session Timeouts

| Phase | Server behavior |
|-------|-----------------|
| Session init | A valid `transcription_session.update` must arrive within **60 s** of connecting, or the server sends a `timeout_error` and closes the stream. |
| Established session | No server-side inactivity timeout. Enforce your own client-side read timeout. |
| After commit | The server sends `input_audio_buffer.committed` once the remaining audio is transcribed, then closes the stream. |

See [README.md](./README.md) for every session field, event and error code, and [data/sample_input.json](./data/sample_input.json) / [data/sample_output.json](./data/sample_output.json) for event payloads.

### 4.4 Prepare Input Audio

Two paths:

- **Smoke test** (next cell): sends 3 seconds of silence to confirm the endpoint is reachable. The transcript is empty.
- **Real audio** (cell after): loads an audio file. Use this to see an actual transcription.

In [ ]:
# --- Connectivity smoke test (silent audio) ---------------------------------
# Emits 3 s of silence, then runs the full transcription_session ->
# input_audio_buffer.commit flow. Expect an EMPTY transcript; the goal is to
# confirm the endpoint routes events correctly end-to-end.

SAMPLE_RATE = 24000
CHUNK_DURATION_MS = 80  # recommended chunk size

smoke_test_samples = np.zeros(SAMPLE_RATE * 3, dtype=np.int16)
smoke_test_bytes = smoke_test_samples.tobytes()
SMOKE_TEST_LANGUAGE = "ja"

print(f"Smoke-test audio ready: {len(smoke_test_bytes)} bytes "
      f"({len(smoke_test_samples) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz, pcm16)")


### 4.4b Load Your Own Audio (recommended)

Replace `WAV_PATH` with any audio file readable by [`soundfile`](https://pysoundfile.readthedocs.io/) (WAV / FLAC / OGG / ...). The cell below checks that the file is mono, decodes it to int16 and prepares it for streaming. The server resamples any rate; 24 kHz is recommended.

```python
WAV_PATH = "path/to/your_audio.wav"
AUDIO_LANGUAGE = "ja"
```

In [ ]:
# ----- Replace THIS with your own audio file -------------------------------
WAV_PATH = "path/to/audio"   # any format supported by libsndfile (WAV, FLAC, OGG, ...)
AUDIO_LANGUAGE = "ja"                   # "ja", "en", "zh", "zh-tw", "ko", "yue", "es" or "vi"
# ---------------------------------------------------------------------------


def load_pcm16(path: str) -> tuple[bytes, int]:
    """Load an audio file and return (int16 little-endian PCM bytes, sample_rate).

    Accepts any format `soundfile` can decode (WAV, FLAC, OGG, etc.). Requires mono;
    the server resamples any sample rate.
    """
    data, samplerate = sf.read(path, dtype="int16")
    if data.ndim != 1:
        raise ValueError(
            f"{path}: expected mono, got {data.shape[1]} channels. "
            "Mix down to mono before passing to Kotoba Multilingual Speech-to-Text (STT) Streaming."
        )
    return data.tobytes(), samplerate


if Path(WAV_PATH).exists():
    real_audio_bytes, real_audio_rate = load_pcm16(WAV_PATH)
    print(
        f"Loaded {WAV_PATH}: {len(real_audio_bytes)} bytes "
        f"({len(real_audio_bytes) / 2 / real_audio_rate:.2f} s @ {real_audio_rate} Hz)"
    )
else:
    real_audio_bytes = None
    real_audio_rate = None
    print(
        f"WAV_PATH={WAV_PATH!r} not found. Update the path above or skip this cell "
        "and run the smoke test only."
    )


### 4.5 Streaming Inference

`transcribe()` opens the HTTP/2 stream, waits for `transcription_session.created`, sends the session settings, streams audio in 80 ms chunks at real-time pace and collects the transcript until `input_audio_buffer.committed`.

Main session options (full table in [README.md](./README.md#session-configuration)):

| Field | Default | Notes |
|-------|---------|-------|
| `input_audio_transcription.target_language` | *(required)* | `"ja"`, `"en"`, `"zh"`, `"zh-tw"`, `"ko"`, `"yue"`, `"es"` or `"vi"` |
| `input_audio_transcription.keywords` | `None` | Up to 1,000 terms of up to 64 characters. |
| `input_audio_transcription.style_preference.human_name` | `"default"` | `"kana"` writes Japanese person names in katakana. |
| `input_audio_transcription.style_preference.verbatim` | `True` | `False` removes fillers and restarts (えーと, あの). Japanese only. |
| `turn_detection` | semantic VAD (`"medium"`) | Omitted = `{"type": "semantic_vad", "eagerness": "medium"}` (recommended). `{"type": "server_vad", "silence_duration_ms": 800}` is also accepted; `false` turns it off. |

Semantic VAD ends a turn soon after a sentence is complete and waits longer while it is not; server VAD ends a turn after every pause of `silence_duration_ms`, which can split a phone number or an address that is read with pauses. For dictated numbers and addresses, use semantic VAD or a longer `silence_duration_ms`.

`transcribe()` returns the concatenation of the `delta` events.

In [ ]:
SESSION_INIT_TIMEOUT = 10.0   # seconds to wait for transcription_session.created/updated (the server allows 60 s)
SESSION_RESULT_TIMEOUT = 60.0  # client-side read timeout while waiting for results
CHUNK_SAMPLES_PCM16 = int(SAMPLE_RATE * CHUNK_DURATION_MS / 1000)
CHUNK_BYTES_PCM16 = CHUNK_SAMPLES_PCM16 * 2  # 2 bytes per int16 sample


async def transcribe(
    endpoint_name: str,
    audio_bytes: bytes,
    *,
    sample_rate: int = SAMPLE_RATE,
    language: str = "ja",
    client_id: str = "nb",
    keywords: list[str] | None = None,
    verbatim: bool | None = None,
    turn_detection: dict | bool | None = None,
) -> str:
    """Transcribe pcm16 mono audio via Kotoba Multilingual Speech-to-Text (STT) Streaming.

    `verbatim=False` (Japanese only) removes fillers. `turn_detection=None` keeps the
    product default (semantic VAD); pass False to turn it off.
    """

    credentials = boto3.Session().get_credentials().get_frozen_credentials()
    config = await AsyncSageMakerRuntimeHTTP2Config.resolve(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        transport=AWSCRTHTTPClient(),  # HTTP/2, needed for bidirectional streaming
    )
    client = AsyncSageMakerRuntimeHTTP2Client(config=config)

    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    # data_type="UTF8" makes SageMaker deliver the payload as a text frame;
    # without it the server receives a binary frame and cannot read the event.
    async def send_event(payload: dict) -> None:
        body = json.dumps(payload).encode("utf-8")
        await stream.input_stream.send(
            RequestStreamEventPayloadPart(
                value=RequestPayloadPart(bytes_=body, data_type="UTF8")
            )
        )

    try:
        _, output_stream = await stream.await_output()

        async def next_event() -> dict:
            message = await output_stream.receive()
            if message is None:
                raise RuntimeError("Stream closed unexpectedly")
            return json.loads(message.value.bytes_.decode("utf-8"))

        # 1) Wait for transcription_session.created
        async with asyncio.timeout(SESSION_INIT_TIMEOUT):
            event = await next_event()
            if event["type"] == "error":
                raise RuntimeError(f"Server error during init: {event}")
            assert event["type"] == "transcription_session.created", event
            print("[session] created")

            # 2) Send session configuration
            await send_event(
                {
                    "type": "transcription_session.update",
                    "session": {
                        "input_audio_format": "pcm16",
                        "input_audio_sample_rate": sample_rate,
                        "input_audio_number_of_channels": 1,
                        "input_audio_transcription": {
                            "language": language,
                            "target_language": language,
                            "keywords": keywords or [],
                            **({"style_preference": {"verbatim": verbatim}} if verbatim is not None else {}),
                        },
                        **({"turn_detection": turn_detection} if turn_detection is not None else {}),
                    },
                }
            )

            # 3) Wait for transcription_session.updated
            event = await next_event()
            if event["type"] == "error":
                raise RuntimeError(f"Server error during update: {event}")
            assert event["type"] == "transcription_session.updated", event
            print("[session] updated")

        # 4) Stream audio chunks (sender) while collecting deltas (receiver)
        transcript_parts: list[str] = []
        chunk_bytes = int(sample_rate * CHUNK_DURATION_MS / 1000) * 2  # pcm16 = 2 bytes/sample

        async def send_audio() -> None:
            for i in range(0, len(audio_bytes), chunk_bytes):
                chunk = audio_bytes[i : i + chunk_bytes]
                await send_event(
                    {
                        "event_id": f"{client_id}_{i // chunk_bytes:07d}",
                        "type": "input_audio_buffer.append",
                        "audio": base64.b64encode(chunk).decode(),
                    }
                )
                await asyncio.sleep(CHUNK_DURATION_MS / 1000)  # simulate real-time pacing
            await send_event({"type": "input_audio_buffer.commit"})

        async def receive_deltas() -> None:
            while True:
                async with asyncio.timeout(SESSION_RESULT_TIMEOUT):
                    event = await next_event()
                etype = event["type"]
                if etype == "conversation.item.input_audio_transcription.delta":
                    delta = event.get("delta", "")
                    transcript_parts.append(delta)
                    print(delta, end="", flush=True)
                elif etype == "conversation.item.created":
                    continue  # informational
                elif etype == "input_audio_buffer.committed":
                    print()  # newline after streaming deltas
                    return
                elif etype == "error":
                    raise RuntimeError(f"Server error: {event}")

        # A TaskGroup cancels the sender when the receiver fails, unlike gather().
        async with asyncio.TaskGroup() as tasks:
            tasks.create_task(send_audio())
            tasks.create_task(receive_deltas())
    finally:
        # Close the stream on every exit, including a failed init.
        await stream.input_stream.close()

    return "".join(transcript_parts)


### 4.6 Run Transcription

**Smoke test first** — confirms the endpoint is healthy. Expect an empty transcript.

In [ ]:
smoke_transcript = await transcribe(
    endpoint_name=ENDPOINT_NAME,
    audio_bytes=smoke_test_bytes,
    sample_rate=SAMPLE_RATE,
    language=SMOKE_TEST_LANGUAGE,
    client_id="smoke",
)
print(f"\nSmoke-test transcript (expected empty): {smoke_transcript!r}")


**Now the real audio** — run this only after the smoke test above completes successfully.

In [ ]:
if real_audio_bytes is None:
    print("Skipping: real audio not loaded. Set WAV_PATH in the loader cell and re-run.")
else:
    transcript = await transcribe(
        endpoint_name=ENDPOINT_NAME,
        audio_bytes=real_audio_bytes,
        sample_rate=real_audio_rate,
        language=AUDIO_LANGUAGE,
        client_id="nb",
    )
    print(f"\n--- Full transcript ---\n{transcript}")


## 5. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | Your kernel is < 3.12. Create a new environment with 3.12+ and re-attach the kernel. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | The HTTP/2 SDK install failed. Re-run cell 2.1, and restart the kernel on SageMaker Studio. |
| `ImportError` from `aws_sdk_sagemaker_runtime_http2` (for example `cannot import name 'AsyncSageMakerRuntimeHTTP2Config'`) | Another version of the HTTP/2 SDK is installed; this notebook needs 0.13. Re-run cell 2.1, which pins it, and restart the kernel. |
| `UnsupportedTransportError`: the transport `AIOHTTPClient` does not support duplex event streaming | The client was built without the HTTP/2 transport. Keep `transport=AWSCRTHTTPClient()` in the config, and re-run cell 2.1 so that `smithy-http[awscrt]` is installed. |
| Error opening the stream, or `TimeoutError` waiting for `transcription_session.created` | The endpoint is not `InService` yet, the endpoint name is wrong, or the region doesn't match the endpoint's. Check with `sagemaker_session.describe_endpoint(EndpointName=ENDPOINT_NAME)`. |
| `TimeoutError` while waiting for results | No event arrived for `SESSION_RESULT_TIMEOUT` (60 s). Check that the endpoint is `InService` and not at its session capacity, or raise the timeout. |
| `Server error during update` with `invalid_parameters` | A session field is invalid; `error.param` names it. Common causes: `target_language` not one of the eight languages, `verbatim: false` with a language other than `"ja"`, `timestamps: true`, `diarization: true`, more than 1,000 keywords or a keyword over 64 characters. |
| `Server error` with `system_busy`, or the stream is refused | The endpoint is at its session capacity. Retry with backoff or raise the instance count. |
| `Server error` with `server_error`, then the stream closes | An internal error ended the session, or audio was sent faster than real time. Pace appends in real time and start a new session. |
| Stream ends after a few minutes with no `error` event | The AWS credentials that signed the stream expired (for example a 15-minute `aws login` or SSO session). Open each stream with credentials that outlive the session. |
| Transcript keeps fillers (えーと, あの) | `style_preference.verbatim` defaults to `true`. Pass `verbatim=False` to `transcribe()` for a clean Japanese transcript. |
| Transcript is empty on real audio | Confirm the audio is speech in `AUDIO_LANGUAGE` and audible, and that the sample rate matches the bytes you send. |
| `AccessDeniedException` at deploy time | Your IAM role lacks the permissions listed under "Required IAM Permissions", or the AWS Marketplace subscription is not complete. |
| Connection closes ~60 s after connect, before `transcription_session.updated` | No valid `transcription_session.update` arrived within the server's 60 s init window. Send it right after `transcription_session.created`. |
| Session opens but sits idle until your client read-timeout fires | Expected when no audio is sent: the server has no inactivity timeout. Close idle sessions from the client. |

## 6. Clean-up

**Important**: Delete the endpoint when done. GPU instances are billed **per hour while the endpoint is active**, even when idle. The audio you stream is also billed per audio hour.

In [ ]:
sagemaker_client = sagemaker_session.sagemaker_client

# Look up the endpoint configuration and the model, so this cell works after a kernel restart too.
config_name = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointConfigName"]
model_names = [
    variant["ModelName"]
    for variant in sagemaker_client.describe_endpoint_config(EndpointConfigName=config_name)[
        "ProductionVariants"
    ]
]

print(f"Deleting endpoint '{ENDPOINT_NAME}', endpoint config '{config_name}' and model(s) {model_names}...")
sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(config_name)
for model_name in model_names:
    sagemaker_session.delete_model(model_name)

print("Endpoint, endpoint config and model deleted.")


### Unsubscribe from Model Package (Optional)

To unsubscribe from the model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba Multilingual Speech-to-Text (STT) Streaming" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, contact kotoba_product+marketplace@kotoba.tech.

## References

- Protocol reference: [README.md](./README.md)
- Event samples: [data/sample_input.json](./data/sample_input.json), [data/sample_output.json](./data/sample_output.json)
- [SageMaker real-time endpoints](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)